# Canada Wildfire 2023 — Interactive Map

This notebook produces an interactive Folium/leafmap visualisation of the 2023
Canadian wildfire season, combining three layers:
- **Heatmap** of all 2023 NFDB fires, weighted by size class
- **City markers** coloured by their fire-impact score
- **Live heatmap** of active fires via the NASA FIRMS API

**Workflow:**
1. Load processed fire & city data
2. Build GeoDataFrames & reproject
3. Spatial join & score cities by nearby fire activity
4. Classify city impact levels
5. Fetch live fire data from NASA FIRMS
6. Assemble map layers & export

## 0  |  Imports & Constants

In [ ]:
# Uncomment on first run to install the leafmap dependency:
# !pip install leafmap

import requests
from datetime import date
from pathlib import Path

import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import leafmap.foliumap as leafmap

# ── File paths ───────────────────────────────────────────────────────────────
DATA_DIR         = Path("../data")
FIRE_CSV_PATH    = DATA_DIR / "processed" / "fire_2023_clean.csv"
CITIES_CSV_PATH  = DATA_DIR / "processed" / "selected_cities_canada.csv"
LIVE_FIRE_PATH   = DATA_DIR / "processed" / "fire_live.csv"
FIRE_GEOJSON_PATH   = DATA_DIR / "processed" / "fire_2023_points.geojson"
CITIES_GEOJSON_PATH = DATA_DIR / "processed" / "selected_cities_canada.geojson"
MAP_OUTPUT_PATH  = Path("../outputs") / "interactive_firemap.html"

# ── Coordinate reference systems ─────────────────────────────────────────────
# EPSG:4326 — geographic degrees; required by Folium and for GeoJSON export.
# EPSG:3347 — Statistics Canada Lambert (metric); required for buffer & sjoin.
CRS_GEOGRAPHIC = "EPSG:4326"
CRS_METRIC     = "EPSG:3347"

# ── Map display ──────────────────────────────────────────────────────────────
# Approximate geographic centre of Canada.
CANADA_MAP_CENTER  = [58.03, -90.82]
CANADA_ZOOM_START  = 4
MAP_TILE_STYLE     = "CartoDB.DarkMatterNoLabels"

# ── Heatmap rendering ────────────────────────────────────────────────────────
HEATMAP_RADIUS      = 13
HEATMAP_BLUR        = 8
HEATMAP_MIN_OPACITY = 0.35

# Inferno-inspired colour ramp (dark → light = low → high intensity).
HEATMAP_GRADIENT = {
    0.00: "#000004",
    0.20: "#420A68",
    0.40: "#6A176E",
    0.60: "#932667",
    0.80: "#DD513A",
    1.00: "#FCFFA4",
}

# ── City buffer & impact classification ──────────────────────────────────────
# Cities accumulate a fire_score from fires within this radius.
CITY_BUFFER_M = 200_000   # 200 km

# fire_score thresholds that define each impact level.
# Scores are sums of class_weight values (1 / 2 / 4 / 16 / 256 per fire).
SCORE_LOW       =  20
SCORE_MODERATE  =  50
SCORE_HIGH      = 256

# Visual encoding: colour and circle radius per impact level.
# Darker blue = higher cumulative fire score near that city.
IMPACT_STYLE = {
    "No impact":        {"color": "#E0FFFF", "radius": 4},
    "Low impact":       {"color": "#87CEFA", "radius": 5},
    "Moderate impact":  {"color": "#1E90FF", "radius": 7},
    "High impact":      {"color": "#0000FF", "radius": 8},
    "Very high impact": {"color": "#191970", "radius": 10},
}

# Legend entries mirror the impact level names above.
LEGEND_IMPACT = {level: style["color"] for level, style in IMPACT_STYLE.items()}

# ── NASA FIRMS live fire API ──────────────────────────────────────────────────
# VIIRS NOAA-20 Near Real-Time data, last 5 days.
FIRMS_API_KEY     = "b73850f35cfea7f219a3969b6df1d9a6"
FIRMS_PRODUCT     = "VIIRS_NOAA20_NRT"
FIRMS_DAY_RANGE   = 5

# Bounding box used to clip the global FIRMS response to Canada.
CANADA_BBOX = {"north": 73.2267, "south": 44.6401, "west": -140.625, "east": -49.2188}

## 1  |  Load Processed Data

In [ ]:
for path in [FIRE_CSV_PATH, CITIES_CSV_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input not found: {path.absolute()}")

fire_2023_df   = pd.read_csv(FIRE_CSV_PATH)
cities_df      = pd.read_csv(CITIES_CSV_PATH)

assert len(fire_2023_df) > 0,  "Fire CSV is empty — re-run the data cleaning notebook."
assert len(cities_df) > 0,     "Cities CSV is empty — re-run the city selection notebook."

print(f"Loaded {len(fire_2023_df):,} fire records and {len(cities_df):,} cities.")

## 2  |  Build GeoDataFrames & Reproject

In [ ]:
# ── 2a  Fire points ──────────────────────────────────────────────────────────
fire_2023_gdf = gpd.GeoDataFrame(
    fire_2023_df,
    geometry=gpd.points_from_xy(fire_2023_df["longitude"], fire_2023_df["latitude"]),
    crs=CRS_GEOGRAPHIC,
)

# Export geographic version for downstream GeoJSON consumers.
FIRE_GEOJSON_PATH.parent.mkdir(parents=True, exist_ok=True)
fire_2023_gdf.to_file(FIRE_GEOJSON_PATH, driver="GeoJSON")

# ── 2b  City points ──────────────────────────────────────────────────────────
cities_gdf = gpd.GeoDataFrame(
    cities_df,
    geometry=gpd.points_from_xy(cities_df["lng"], cities_df["lat"]),
    crs=CRS_GEOGRAPHIC,
)

CITIES_GEOJSON_PATH.parent.mkdir(parents=True, exist_ok=True)
cities_gdf.to_file(CITIES_GEOJSON_PATH, driver="GeoJSON")

# ── 2c  Reproject both layers to metric CRS for buffer & spatial join ────────
# Distance-based operations (buffer, sjoin) require a metric projection.
cities_metric_gdf    = cities_gdf.to_crs(CRS_METRIC)
fire_2023_metric_gdf = fire_2023_gdf.to_crs(CRS_METRIC)

assert cities_metric_gdf.crs == fire_2023_metric_gdf.crs, (
    "CRS mismatch between city and fire layers — spatial join would be invalid."
)
print("Both layers reprojected to EPSG:3347.")

## 3  |  Score Cities by Nearby Fire Activity

In [ ]:
# ── 3a  Buffer cities ────────────────────────────────────────────────────────
cities_buffered_gdf = cities_metric_gdf.copy()
cities_buffered_gdf["geometry"] = cities_buffered_gdf.geometry.buffer(CITY_BUFFER_M)

# ── 3b  Spatial join: find all fires inside each city's buffer ───────────────
# Left join preserves cities that have no fires within their buffer.
joined_df = gpd.sjoin(
    cities_buffered_gdf,
    fire_2023_metric_gdf,
    how="left",
    predicate="contains",
)

# ── 3c  Aggregate fire scores per city ──────────────────────────────────────
# class_weight encodes fire severity; summing it gives a composite impact score.
fire_score_df = (
    joined_df
    .groupby("city")["class_weight"]
    .sum()
    .reset_index()
    .rename(columns={"class_weight": "fire_score"})
)

# Merge scores back and return to geographic CRS for Folium rendering.
cities_scored_gdf = (
    cities_metric_gdf
    .merge(fire_score_df, on="city", how="left")
    .to_crs(CRS_GEOGRAPHIC)
)

# Cities with no fires in their buffer receive a score of 0.
cities_scored_gdf["fire_score"] = cities_scored_gdf["fire_score"].fillna(0)

print(f"Cities with fire_score > 0: {(cities_scored_gdf['fire_score'] > 0).sum()}")

## 4  |  Classify City Impact Levels

In [ ]:
def classify_impact(fire_score: float) -> str:
    """
    Classifies a city's wildfire impact from its cumulative fire score.

    Thresholds are defined by the module-level constants SCORE_LOW,
    SCORE_MODERATE, and SCORE_HIGH, which are sums of fire class_weight
    values (1 / 2 / 4 / 16 / 256) from the NFDB cleaning step.

    Parameters
    ----------
    fire_score : float
        Cumulative class_weight of all fires within CITY_BUFFER_M of a city.

    Returns
    -------
    str
        One of: 'No impact', 'Low impact', 'Moderate impact',
        'High impact', 'Very high impact'.
    """
    if fire_score == 0:
        return "No impact"
    if fire_score < SCORE_LOW:
        return "Low impact"
    if fire_score < SCORE_MODERATE:
        return "Moderate impact"
    if fire_score < SCORE_HIGH:
        return "High impact"
    return "Very high impact"


cities_scored_gdf["impact"] = cities_scored_gdf["fire_score"].apply(classify_impact)

print(cities_scored_gdf["impact"].value_counts())

## 5  |  Fetch Live Fire Data (NASA FIRMS)

In [ ]:
today     = date.today()
api_url   = (
    f"https://firms.modaps.eosdis.nasa.gov/usfs/api/area/csv/"
    f"{FIRMS_API_KEY}/{FIRMS_PRODUCT}/world/{FIRMS_DAY_RANGE}/{today}"
)

response = requests.get(api_url, timeout=30)

if response.status_code != 200:
    raise ConnectionError(
        f"NASA FIRMS API returned status {response.status_code}. "
        "Check your API key or network connection."
    )

fire_live_df = pd.read_csv(api_url)

assert len(fire_live_df) > 0, "NASA FIRMS returned an empty dataset."

# Clip the global response to Canada's bounding box to reduce rendering load.
fire_live_canada_df = fire_live_df[
    (fire_live_df["latitude"]  >= CANADA_BBOX["south"]) &
    (fire_live_df["latitude"]  <= CANADA_BBOX["north"]) &
    (fire_live_df["longitude"] >= CANADA_BBOX["west"])  &
    (fire_live_df["longitude"] <= CANADA_BBOX["east"])
]

# Save the raw global response for reproducibility; the in-memory copy is clipped.
LIVE_FIRE_PATH.parent.mkdir(parents=True, exist_ok=True)
fire_live_df.to_csv(LIVE_FIRE_PATH, index=False)

print(f"Live fire records for Canada: {len(fire_live_canada_df):,}")

## 6  |  Assemble Map Layers & Export

In [ ]:
# ── 6a  Base map ─────────────────────────────────────────────────────────────
canada_map = leafmap.Map(
    location=CANADA_MAP_CENTER,
    zoom_start=CANADA_ZOOM_START,
    tiles=MAP_TILE_STYLE,
)

# ── 6b  NFDB 2023 weighted heatmap ───────────────────────────────────────────
# Each point is weighted by class_weight so mega-fires dominate the heat signal.
heat_data = [
    [point.xy[1][0], point.xy[0][0], weight]
    for point, weight in zip(fire_2023_gdf.geometry, fire_2023_gdf["class_weight"])
]

plugins.HeatMap(
    heat_data,
    radius=HEATMAP_RADIUS,
    blur=HEATMAP_BLUR,
    min_opacity=HEATMAP_MIN_OPACITY,
    name="Wildfires in 2023",
    gradient=HEATMAP_GRADIENT,
).add_to(canada_map)

# ── 6c  City impact markers ──────────────────────────────────────────────────
impacted_cities_layer = folium.FeatureGroup(name="Impacted Cities in 2023")

for _, row in cities_scored_gdf.iterrows():
    style = IMPACT_STYLE[row["impact"]]

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=style["radius"],
        color="gray",
        weight=0.9,
        fill=True,
        fill_color=style["color"],
        fill_opacity=0.95,
        tooltip=(
            f"Name: {row['city']} <br>"
            f"Country: {row['country']}<br>"
            f"Population: {row['population']} <br>"
            f"Impact: {row['impact']}"
        ),
    ).add_to(impacted_cities_layer)

impacted_cities_layer.add_to(canada_map)

# ── 6d  NASA FIRMS live heatmap (hidden by default) ──────────────────────────
live_heat_data = [
    [row["latitude"], row["longitude"]]
    for _, row in fire_live_canada_df.iterrows()
]

plugins.HeatMap(
    live_heat_data,
    radius=HEATMAP_RADIUS,
    blur=5,
    min_opacity=0.25,
    name="Live wildfire",
    show=False,   # off by default so it does not visually compete with the 2023 layer
    gradient=HEATMAP_GRADIENT,
).add_to(canada_map)

# ── 6e  Legend & layer control ───────────────────────────────────────────────
canada_map.add_legend(title="Fire impact on city", legend_dict=LEGEND_IMPACT)
folium.LayerControl(collapsed=False).add_to(canada_map)

# ── 6f  Export ───────────────────────────────────────────────────────────────
MAP_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
canada_map.save(MAP_OUTPUT_PATH)

print(f"Map saved → {MAP_OUTPUT_PATH}")

## 7  |  Display Map

In [ ]:
canada_map